# 5 Game-Changing Free Tools to Enhance Your Data Science Portfolio

Local runnable Voilà companion. No cloud accounts, keys, payments or public publishing. The fictional three-project dataset is a supplied teaching fixture. Original installation/launch fragments remain verbatim in the nonexecuting appendix.

In [ ]:
import sys, html, json, tempfile, socket, subprocess, time, secrets, os
from pathlib import Path
from importlib.metadata import version
import urllib.request
import nbformat
import ipywidgets as widgets
from IPython.display import HTML, display
print("Python:", sys.version.split()[0])
print("Voilà:", version("voila"), "ipywidgets:", version("ipywidgets"))

In [ ]:
PROJECTS = [
    {"title": "Sales dashboard", "topic": "Analytics", "summary": "A fictional monthly sales exploration."},
    {"title": "Image classifier", "topic": "Machine Learning", "summary": "A fictional image-model portfolio entry."},
    {"title": "Forecast notebook", "topic": "Machine Learning", "summary": "A fictional time-series project entry."}]
def build_dashboard(projects):
    selector = widgets.Dropdown(options=["All", "Analytics", "Machine Learning"], description="Topic:")
    cards = widgets.HTML()
    def render(change=None):
        chosen = [p for p in projects if selector.value == "All" or p["topic"] == selector.value]
        cards.value = "<p>Projects: " + str(len(chosen)) + "</p>" + "".join(
            "<article><h3>" + html.escape(p["title"]) + "</h3><p>" + html.escape(p["summary"]) + "</p></article>"
            for p in chosen)
    selector.observe(render, names="value")
    render()
    return widgets.VBox([selector, cards]), selector, cards
dashboard, selector, cards = build_dashboard(PROJECTS)
display(HTML("<h2>Portfolio Projects</h2>"))
display(dashboard)
print("Three fictional project cards ready.")

In [ ]:
assert "Projects: 3" in cards.value
selector.value = "Analytics"
assert "Projects: 1" in cards.value and "Sales dashboard" in cards.value and "Image classifier" not in cards.value
selector.value = "Machine Learning"
assert "Projects: 2" in cards.value and "Forecast notebook" in cards.value
selector.value = "All"
other, other_selector, other_cards = build_dashboard(PROJECTS)
other_selector.value = "Analytics"
assert selector.value == "All" and "Projects: 3" in cards.value
unsafe, unsafe_selector, unsafe_cards = build_dashboard([{"title": "<script>alert(1)</script>", "topic": "Analytics", "summary": "<img src=x onerror=alert(1)>"}])
assert "<script>" not in unsafe_cards.value and "<img" not in unsafe_cards.value
assert "&lt;script&gt;" in unsafe_cards.value
print("PASS: actual widget observers filter 3→1→2→3 cards, instance state is independent and supplied text is escaped.")

In [ ]:
work_dir = Path(tempfile.mkdtemp(prefix="tdb-portfolio-"))
from ipykernel.kernelspec import install
os.environ["JUPYTER_PATH"] = str(work_dir / "share/jupyter")
install(prefix=str(work_dir), kernel_name="tdb-portfolio", display_name="Portfolio demo")
dashboard_source = 'import sys, html, json, tempfile, socket, subprocess, time, secrets, os\nfrom pathlib import Path\nfrom importlib.metadata import version\nimport urllib.request\nimport nbformat\nimport ipywidgets as widgets\nfrom IPython.display import HTML, display\nprint("Python:", sys.version.split()[0])\nprint("Voilà:", version("voila"), "ipywidgets:", version("ipywidgets"))\n\nPROJECTS = [\n    {"title": "Sales dashboard", "topic": "Analytics", "summary": "A fictional monthly sales exploration."},\n    {"title": "Image classifier", "topic": "Machine Learning", "summary": "A fictional image-model portfolio entry."},\n    {"title": "Forecast notebook", "topic": "Machine Learning", "summary": "A fictional time-series project entry."}]\ndef build_dashboard(projects):\n    selector = widgets.Dropdown(options=["All", "Analytics", "Machine Learning"], description="Topic:")\n    cards = widgets.HTML()\n    def render(change=None):\n        chosen = [p for p in projects if selector.value == "All" or p["topic"] == selector.value]\n        cards.value = "<p>Projects: " + str(len(chosen)) + "</p>" + "".join(\n            "<article><h3>" + html.escape(p["title"]) + "</h3><p>" + html.escape(p["summary"]) + "</p></article>"\n            for p in chosen)\n    selector.observe(render, names="value")\n    render()\n    return widgets.VBox([selector, cards]), selector, cards\ndashboard, selector, cards = build_dashboard(PROJECTS)\ndisplay(HTML("<h2>Portfolio Projects</h2>"))\ndisplay(dashboard)\nprint("Three fictional project cards ready.")'
demo_notebook = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(dashboard_source)],
    metadata={"kernelspec": {"name": "tdb-portfolio", "display_name": "Portfolio demo", "language": "python"}})
nbformat.write(demo_notebook, work_dir / "portfolio_dashboard.ipynb")
print("Created a runnable, output-free portfolio_dashboard.ipynb in a new temporary directory.")

In [ ]:
def free_local_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]
def check_server(command, paths, env, token=None):
    with (work_dir / ("server-" + str(command[2]).replace("/", "_") + ".log")).open("w") as log:
        server = subprocess.Popen(command, cwd=work_dir, env=env, stdout=log, stderr=log)
        try:
            for url, marker in paths:
                for _ in range(150):
                    if server.poll() is not None:
                        raise RuntimeError("The local server exited before readiness")
                    headers = {} if token is None else {"Authorization": "token " + token}
                    try:
                        with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as response:
                            body = response.read().decode()
                            assert response.status == 200 and marker.lower() in body.lower()
                            break
                    except OSError:
                        time.sleep(0.2)
                else:
                    raise TimeoutError("Local dashboard did not become ready")
        finally:
            server.terminate()
            server.wait(timeout=20)
child_env = dict(os.environ, JUPYTER_PATH=str(work_dir / "share/jupyter"))
port = free_local_port()
standalone_command = [sys.executable, "-m", "voila", "portfolio_dashboard.ipynb", "--no-browser", "--Voila.ip=127.0.0.1", "--port=" + str(port)]
check_server(standalone_command, [(f"http://127.0.0.1:{port}/", "Portfolio Projects")], child_env)
print("Standalone Voilà: HTTP 200; real notebook kernel executed and dashboard rendered.")
port = free_local_port()
token = secrets.token_urlsafe(32)
child_env["JUPYTER_TOKEN"] = token
extension_command = [sys.executable, "-m", "jupyter_server", "--ServerApp.ip=127.0.0.1", "--ServerApp.port=" + str(port), "--ServerApp.port_retries=0", "--ServerApp.open_browser=False", "--ServerApp.root_dir=" + str(work_dir), "--ServerApp.jpserver_extensions={'voila.server_extension': True}"]
check_server(extension_command, [
    (f"http://127.0.0.1:{port}/voila", "voila"),
    (f"http://127.0.0.1:{port}/voila/render/portfolio_dashboard.ipynb", "Portfolio Projects")], child_env, token)
del child_env["JUPYTER_TOKEN"]
del token
print("Authenticated Jupyter server extension: listing/render routes HTTP 200; no public tunnel or cloud publishing.")

In [ ]:
assert len(PROJECTS) == 3 and selector.value == "All"
assert len(demo_notebook.cells) == 1 and demo_notebook.cells[0].outputs == []
assert demo_notebook.cells[0].source == dashboard_source
assert not any("api_key" in p for p in PROJECTS)
print("PASS: original standalone and server-extension flows run on loopback with real Voilà kernels; widget state and escaping checks pass. No mamba installation or cloud-account action is claimed.")

## Original source code — nonexecuting appendix

### Original block 1

```text
mamba install -c conda-forge voila
```

### Original block 2

```text
pip install voila
```

### Original block 3

```text
voila <path-to-notebook> <options>
```

### Original block 4

```text
<url-of-my-server>/voila
```